# Kernel Profiling

Profiles the pure-PyTorch reference implementation across shapes and
produces the roofline table for `docs/kernels.md`.

GPU profiling (Triton kernel) requires a CUDA device; CPU reference
profiling works on any machine.

In [ ]:
import sys; sys.path.insert(0, '..')
import time
import torch
from flashspec.kernels._reference import verify_tokens_reference
from flashspec.utils.device import set_seed, is_cuda_available
print(f'CUDA available: {is_cuda_available()}')

In [ ]:
# ── Profile shapes ────────────────────────────────────────────────────────────
SHAPES = [
    ('B=1  γ=4  V=32k',  1,  4, 32_000),
    ('B=8  γ=4  V=32k',  8,  4, 32_000),
    ('B=1  γ=8  V=32k',  1,  8, 32_000),
    ('B=32 γ=4  V=32k', 32,  4, 32_000),
]
N_WARMUP = 10
N_STEPS  = 100

def profile_reference(B, gamma, V, n_warmup=N_WARMUP, n_steps=N_STEPS):
    set_seed(0)
    dlp = torch.randn(B, gamma, V).log_softmax(-1)
    tlp = torch.randn(B, gamma, V).log_softmax(-1)
    u   = torch.rand(B, gamma)
    ids = torch.randint(0, V, (B, gamma))
    for _ in range(n_warmup):
        verify_tokens_reference(dlp, tlp, u, ids)
    lats = []
    for _ in range(n_steps):
        t0 = time.perf_counter()
        verify_tokens_reference(dlp, tlp, u, ids)
        lats.append((time.perf_counter() - t0) * 1000)
    return sum(lats)/len(lats), sorted(lats)[int(0.99*len(lats))]

print(f'{"Shape":<24} {"Mean (ms)":>10} {"p99 (ms)":>10}')
print('-' * 46)
for label, B, gamma, V in SHAPES:
    mean_ms, p99_ms = profile_reference(B, gamma, V)
    print(f'{label:<24} {mean_ms:>10.3f} {p99_ms:>10.3f}')

In [ ]:
# ── GPU Triton profiling (skipped if no CUDA) ─────────────────────────────────
if not is_cuda_available():
    print('No CUDA device — skipping Triton profiling.')
else:
    from flashspec.kernels import verify_tokens
    device = torch.device('cuda')
    print(f'\nTriton kernel on {torch.cuda.get_device_name(0)}')
    print(f'{"Shape":<24} {"Ref (ms)":>10} {"Triton (ms)":>12} {"Speedup":>10}')
    print('-' * 58)
    for label, B, gamma, V in SHAPES:
        ref_mean, _ = profile_reference(B, gamma, V)
        set_seed(0)
        dlp = torch.randn(B, gamma, V).log_softmax(-1).to(device)
        tlp = torch.randn(B, gamma, V).log_softmax(-1).to(device)
        u   = torch.rand(B, gamma, device=device)
        ids = torch.randint(0, V, (B, gamma), device=device)
        for _ in range(N_WARMUP):
            verify_tokens(dlp, tlp, u, ids)
        lats = []
        for _ in range(N_STEPS):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            verify_tokens(dlp, tlp, u, ids)
            torch.cuda.synchronize()
            lats.append((time.perf_counter() - t0) * 1000)
        triton_mean = sum(lats)/len(lats)
        print(f'{label:<24} {ref_mean:>10.3f} {triton_mean:>12.3f} {ref_mean/triton_mean:>9.1f}×')